# Bài 1.2: Tách từ Tiếng Việt (Word Segmentation), POS Tagging & Dependency Parsing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vietnamese-nlp-mastery/vietnamese-nlp-mastery/blob/main/01_Foundations_Preprocessing/02_word_segmentation_underthesea_vncorenlp.ipynb)

> **Mục tiêu bài học:**
> 1. Hiểu rõ tại sao khoảng trắng trong tiếng Việt phân tách **Âm tiết (Syllables / Tiếng)** chứ không phải **Từ (Words)**, và hiện tượng nhập nhằng ranh giới từ (Word Boundary Ambiguity).
> 2. Thực hành và so sánh 3 bộ công cụ tách từ kinh điển: **`pyvi`**, **`underthesea`**, và **`VnCoreNLP` (RDRSegmenter)**.
> 3. Thực hành Gán nhãn từ loại (**POS Tagging**) và Phân tích cú pháp phụ thuộc (**Dependency Parsing**).
> 4. Nắm vững quy tắc thực chiến: **Khi nào BẮT BUỘC phải tách từ** (`PhoBERT`, `PhoW2V`, `BM25`) và **khi nào KHÔNG tách từ** (`ViT5`, `Qwen`, `Sailor2`, `BGE-M3`).

## 1. Tại sao Tiếng Việt cần Tách từ (Word Segmentation)?

Trong tiếng Anh, khoảng trắng phân tách trực tiếp các từ (`"student"`, `"biology"`). Ngược lại, trong tiếng Việt, khoảng trắng dùng để phân tách các **âm tiết (tiếng)**:
- `"học"` (to study) + `"sinh"` (to be born / life) -> Ghép lại có thể là **`"học_sinh"`** (student) hoặc **`"sinh_học"`** (biology).

Xét câu kinh điển:
> **`"Học sinh học sinh học"`**
- Nếu tách theo khoảng trắng (Syllable-level): chỉ có 2 token duy nhất là `"học"` (xuất hiện 2 lần) và `"sinh"` (xuất hiện 3 lần) -> Mô hình Bag-of-Words / TF-IDF mất hoàn toàn ngữ nghĩa!
- Nếu tách từ chuẩn (Word-level): **`["Học_sinh", "học", "sinh_học"]`** (*Students study biology*) -> 3 đơn vị từ vựng hoàn toàn rõ nghĩa.

In [1]:
# Minh họa sự khác biệt giữa tách theo âm tiết (Syllable) và tách theo từ ghép (Compound Word)
ambiguous_sentences = [
    ("Học sinh học sinh học", ["Học_sinh", "học", "sinh_học"]),
    ("Ông già đi nhanh quá", ["Ông_già", "đi", "nhanh", "quá"]),
    ("Bàn là công cụ làm phẳng quần áo", ["Bàn_là", "là", "công_cụ", "làm", "phẳng", "quần_áo"]),
]

for raw, gold_words in ambiguous_sentences:
    syllables = raw.split()
    print(f"Câu gốc        : {raw}")
    print(f"Tách âm tiết   : {syllables} ({len(syllables)} âm tiết)")
    print(f"Tách từ chuẩn  : {gold_words} ({len(gold_words)} từ)")
    print("-" * 65)

Câu gốc        : Học sinh học sinh học
Tách âm tiết   : ['Học', 'sinh', 'học', 'sinh', 'học'] (5 âm tiết)
Tách từ chuẩn  : ['Học_sinh', 'học', 'sinh_học'] (3 từ)
-----------------------------------------------------------------
Câu gốc        : Ông già đi nhanh quá
Tách âm tiết   : ['Ông', 'già', 'đi', 'nhanh', 'quá'] (5 âm tiết)
Tách từ chuẩn  : ['Ông_già', 'đi', 'nhanh', 'quá'] (4 từ)
-----------------------------------------------------------------
Câu gốc        : Bàn là công cụ làm phẳng quần áo
Tách âm tiết   : ['Bàn', 'là', 'công', 'cụ', 'làm', 'phẳng', 'quần', 'áo'] (8 âm tiết)
Tách từ chuẩn  : ['Bàn_là', 'là', 'công_cụ', 'làm', 'phẳng', 'quần_áo'] (6 từ)
-----------------------------------------------------------------


## 2. So sánh 3 Bộ công cụ Tách từ: `pyvi`, `underthesea`, và `VnCoreNLP`

| Tiêu chí | `pyvi` | `underthesea` | `VnCoreNLP` (`py_vncorenlp`) |
| :--- | :--- | :--- | :--- |
| **Thuật toán lõi** | Linear-chain CRF | CRF + Deep Learning | **RDRSegmenter** (Ripple Down Rules) |
| **Ngôn ngữ cài đặt** | Pure Python / Cython | Pure Python | Java (gọi qua wrapper Python) |
| **Tốc độ xử lý** | Rất nhanh (~15,000 câu/s) | Trung bình (~2,500 câu/s) | Nhanh (~12,000 câu/s sau khi khởi tạo JVM) |
| **Độ chính xác (F1 VLSP)** | ~97.0% | ~97.5% | **~97.9% (Chuẩn huấn luyện của PhoBERT)** |
| **Trường hợp khuyên dùng** | Tiền xử lý nhanh cho TF-IDF, BM25, Web App nhẹ | Pipeline tổng hợp (Tách từ, POS, NER, Dependency) | **Bắt buộc khi dùng `PhoBERT`, `PhoW2V`, `PhoNLP`** |

In [2]:
import time


def benchmark_segmenters(sentences: list[str], n_repeats: int = 100) -> None:
    corpus = sentences * n_repeats
    print(f"Đang chạy benchmark trên {len(corpus)} câu...\n")

    # 1. PyVi
    try:
        from pyvi import ViTokenizer

        t0 = time.perf_counter()
        pyvi_out = [ViTokenizer.tokenize(s) for s in corpus]
        dt = time.perf_counter() - t0
        print(f"[pyvi]        Tốc độ: {len(corpus) / dt:,.0f} câu/s | Mẫu: {pyvi_out[0]}")
    except ImportError:
        print("[pyvi]        Chưa cài đặt (`pip install pyvi`)")

    # 2. Underthesea
    try:
        from underthesea import word_tokenize

        t0 = time.perf_counter()
        uts_out = [word_tokenize(s, format="text") for s in corpus]
        dt = time.perf_counter() - t0
        print(f"[underthesea] Tốc độ: {len(corpus) / dt:,.0f} câu/s | Mẫu: {uts_out[0]}")
    except ImportError:
        print("[underthesea] Chưa cài đặt (`pip install underthesea`)")


test_sentences = [
    "Viện Nghiên cứu Trí tuệ Nhân tạo VinAI vừa công bố mô hình ngôn ngữ PhoBERT.",
    "Học sinh học sinh học tại trường trung học phổ thông chuyên Hà Nội Amsterdam.",
    "Giảng viên hướng dẫn sinh viên thực hành xử lý ngôn ngữ tự nhiên rất nhiệt tình.",
]

benchmark_segmenters(test_sentences, n_repeats=50)

Đang chạy benchmark trên 150 câu...

[pyvi]        Tốc độ: 14,820 câu/s | Mẫu: Viện Nghiên_cứu Trí_tuệ Nhân_tạo VinAI vừa công_bố mô_hình ngôn_ngữ PhoBERT .
[underthesea] Tốc độ: 2,410 câu/s | Mẫu: Viện Nghiên_cứu Trí_tuệ Nhân_tạo VinAI vừa công_bố mô_hình ngôn_ngữ PhoBERT .


## 3. Gán nhãn Từ loại (POS Tagging) & Phân tích Cú pháp (Dependency Parsing)

Bên cạnh tách từ, **`underthesea`** và **`VnCoreNLP`** cung cấp pipeline phân tích cú pháp hoàn chỉnh:
- **POS Tagging**: Xác định danh từ (`N`), danh từ riêng (`Np`), động từ (`V`), tính từ (`A`), giới từ (`E`)...
- **Dependency Parsing**: Xác định quan hệ phụ thuộc giữa các từ trong câu (`sub` - chủ ngữ, `dob` - tân ngữ trực tiếp, `root` - động từ chính).

In [3]:
sentence = "Mô hình PhoBERT cải thiện đáng kể độ chính xác phân loại văn bản tiếng Việt."

try:
    from underthesea import dependency_parse, pos_tag

    print("=== POS TAGGING (Underthesea) ===")
    for word, tag in pos_tag(sentence):
        print(f"  {word:<15} -> {tag}")

    print("\n=== DEPENDENCY PARSING (Underthesea) ===")
    for word, head, rel in dependency_parse(sentence):
        print(f"  {word:<15} | head={head:<3} | relation={rel}")
except ImportError:
    print("Cài đặt underthesea (`pip install underthesea`) để chạy trực tiếp POS & Dependency Parsing.")

=== POS TAGGING (Underthesea) ===
  Mô hình         -> N
  PhoBERT         -> Np
  cải thiện       -> V
  đáng kể         -> A
  độ              -> N
  chính xác       -> A
  phân loại       -> V
  văn bản         -> N
  tiếng           -> N
  Việt            -> Np
  .               -> CH

=== DEPENDENCY PARSING (Underthesea) ===
  Mô hình         | head=3   | relation=sub
  PhoBERT         | head=1   | relation=nmod
  cải thiện       | head=0   | relation=root
  đáng kể         | head=3   | relation=adv
  độ              | head=3   | relation=dob
  chính xác       | head=5   | relation=nmod
  phân loại       | head=5   | relation=nmod
  văn bản         | head=7   | relation=dob
  tiếng           | head=8   | relation=nmod
  Việt            | head=9   | relation=nmod
  .               | head=3   | relation=punct


## 4. Tích hợp `VnCoreNLP` (`py_vncorenlp`) Chuẩn cho `PhoBERT`

> **Quy tắc vàng khi dùng `PhoBERT` / `PhoNLP`:**
> `PhoBERT` được tiền huấn luyện trên 20GB văn bản đã qua bước tách từ của **RDRSegmenter (`VnCoreNLP`)** (nối từ ghép bằng dấu gạch dưới `_`, ví dụ `nghiên_cứu`). Nếu bạn truyền văn bản chưa tách từ vào `PhoBERT`, hiệu năng sẽ giảm đáng kể!

```python
import py_vncorenlp

# Tải tự động model VnCoreNLP (cần cài đặt Java JDK >= 1.8)
py_vncorenlp.download_model(save_dir="./vncorenlp_models")

# Chỉ load wseg (Word Segmenter) để tối ưu tốc độ và RAM khi tiền xử lý cho PhoBERT
rdrsegmenter = py_vncorenlp.VnCoreNLP(
    annotators=["wseg"],
    save_dir="./vncorenlp_models"
)

raw_text = "Ông Nguyễn Khắc Chúc đang làm việc tại Đại học Quốc gia Hà Nội."
segmented_sentences = rdrsegmenter.word_segment(raw_text)
print(segmented_sentences[0])
# Output: "Ông Nguyễn_Khắc_Chúc đang làm_việc tại Đại_học Quốc_gia Hà_Nội ."
```

### Bảng Tổng Kết: Khi nào Cần và Không Cần Tách Từ (`_` nối từ ghép)?

| Mô hình / Kỹ thuật | Có cần nối từ ghép (`_`) không? | Lý do |
| :--- | :---: | :--- |
| **TF-IDF / BM25 Tiếng Việt** | **CÓ** | Tránh tách rời các âm tiết vô nghĩa, giữ đúng từ khóa ghép (`học_sinh` vs `sinh_học`). |
| **PhoW2V (Word-level)** | **CÓ** | Từ điển xây dựng trên các từ ghép đã tách bằng `VnCoreNLP`. |
| **PhoBERT (`v1` & `v2`), PhoNLP** | **BẮT BUỘC CÓ** | BPE Tokenizer của PhoBERT kỳ vọng đầu vào đã được tách từ bằng RDRSegmenter. |
| **ViT5 (`VietAI/vit5-base`)** | **KHÔNG** | Sử dụng SentencePiece trực tiếp trên văn bản tiếng Việt tự nhiên. |
| **BGE-M3, Halong Embedding** | **KHÔNG** | Mô hình đa ngôn ngữ hiện đại sử dụng subword tokenizer trên văn bản nguyên bản. |
| **Qwen2.5, Sailor2, Llama-3, Gemini** | **KHÔNG** | Byte-level BPE Tokenizer được huấn luyện trên văn bản tự nhiên không nối `_`. |